In [1]:
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"

  Cloning https://github.com/unslothai/unsloth.git to /tmp/pip-install-pexp9x0v/unsloth_9ff015d02a404a22bc7a4988ebf6911f
  Running command git clone --filter=blob:none --quiet https://github.com/unslothai/unsloth.git /tmp/pip-install-pexp9x0v/unsloth_9ff015d02a404a22bc7a4988ebf6911f
  Resolved https://github.com/unslothai/unsloth.git to commit 06daf28c8b79782375bb7e17a830b11266407bc9
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.1/59.1 MB 16.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 46.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 290.7/290.7 kB 30.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 180.7/180.7 kB 19.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.7/47.7 MB 20.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.2/7.2 MB 147.4 MB/s eta 0:00:00

In [15]:
!pip install --no-deps xformers peft accelerate bitsandbytes
!pip uninstall -y trl
!pip install trl==0.24.0

Found existing installation: trl 0.8.6
Uninstalling trl-0.8.6:
  Successfully uninstalled trl-0.8.6
  Using cached trl-0.24.0-py3-none-any.whl.metadata (11 kB)
Using cached trl-0.24.0-py3-none-any.whl (423 kB)


In [3]:
import json
import os

# --- Configuration ---
input_file = "/content/medquad_converted.jsonl"
output_file = "/content/cleaned_dataset.jsonl"

def clean_dataset(input_path, output_path):
    print(f"Reading from {input_path}...")
    valid_rows = []
    errors = 0

    # Check if file exists
    if not os.path.exists(input_path):
        print(f"Error: File {input_path} not found.")
        return

    with open(input_path, 'r', encoding='utf-8') as infile:
        for line_num, line in enumerate(infile):
            try:
                data = json.loads(line)

                # Check 1: Must have "messages" key
                if "messages" not in data:
                    print(f"Skipping line {line_num}: Missing 'messages' key.")
                    errors += 1
                    continue

                messages = data["messages"]

                # Check 2: Messages must be a list
                if not isinstance(messages, list):
                    print(f"Skipping line {line_num}: 'messages' is not a list.")
                    errors += 1
                    continue

                # Check 3: Verify structure of each message
                is_valid_structure = True
                for msg in messages:
                    if "role" not in msg or "content" not in msg:
                        is_valid_structure = False
                        break
                    if not isinstance(msg["content"], str) or not msg["content"].strip():
                        is_valid_structure = False # Skip empty content
                        break

                if not is_valid_structure:
                    print(f"Skipping line {line_num}: Invalid message structure or empty content.")
                    errors += 1
                    continue

                # If all checks pass, add to list
                valid_rows.append(data)

            except json.JSONDecodeError:
                print(f"Skipping line {line_num}: Invalid JSON.")
                errors += 1

    # Remove Duplicates (based on the content of the messages)
    unique_rows = []
    seen_hashes = set()
    for row in valid_rows:
        # Create a string representation to hash
        row_str = json.dumps(row["messages"], sort_keys=True)
        if row_str not in seen_hashes:
            seen_hashes.add(row_str)
            unique_rows.append(row)

    # Save to new file
    with open(output_path, 'w', encoding='utf-8') as outfile:
        for row in unique_rows:
            outfile.write(json.dumps(row) + '\n')

    print("-" * 30)
    print(f"Cleaning Complete.")
    print(f"Original Rows: {len(valid_rows) + errors}")
    print(f"Valid Rows: {len(valid_rows)}")
    print(f"Unique Rows (after deduplication): {len(unique_rows)}")
    print(f"Saved to: {output_path}")

# Run the function
clean_dataset(input_file, output_file)

Reading from /content/medquad_converted.jsonl...
------------------------------
Cleaning Complete.
Original Rows: 15578
Valid Rows: 15578
Unique Rows (after deduplication): 15530
Saved to: /content/cleaned_dataset.jsonl


In [4]:
from unsloth import FastLanguageModel
from unsloth.chat_templates import get_chat_template
import torch
from datasets import load_dataset
from trl import SFTTrainer
from transformers import TrainingArguments

# --- 1. Configuration ---
# ERROR PREVENTION: Ensure this model name is correct.
# If using IBM Granite, it might be "unsloth/granite-20b-code-instruct-bnb-4bit"
model_name = "unsloth/gpt-oss-20b-unsloth-bnb-4bit"
new_model_name = "my_finetuned_model_lora" # Name for the saved adapters
max_seq_length = 2048
dtype = None
load_in_4bit = True

# --- 2. Load Model & Tokenizer ---
print(f"Loading model: {model_name}...")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = model_name,
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

# --- 3. Add LoRA Adapters ---
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
)

# --- 4. Load & Format Dataset ---
# Setup Chat Template (Llama-3 style is standard/robust)
tokenizer = get_chat_template(
    tokenizer,
    chat_template = "llama-3",
    mapping = {"role" : "role", "content" : "content", "user" : "user", "assistant" : "assistant"},
)

# Load the CLEANED dataset from Part 1
dataset = load_dataset("json", data_files="/content/cleaned_dataset.jsonl", split="train")

def formatting_prompts_func(examples):
    convos = examples["messages"]
    texts = [tokenizer.apply_chat_template(convo, tokenize=False, add_generation_prompt=False) for convo in convos]
    return { "text" : texts }

dataset = dataset.map(formatting_prompts_func, batched = True)

# --- 5. Configure Trainer ---
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    packing = False,
    args = TrainingArguments(
        per_device_train_batch_size = 1, # Keep low for 20B models on limited VRAM
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        max_steps = 300, # Adjust based on dataset size
        learning_rate = 2e-4,
        fp16 = not torch.cuda.is_bf16_supported(),
        bf16 = torch.cuda.is_bf16_supported(),
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "model_output",
        report_to = "none", # Disable wandb/tensorboard for clean output
    ),
)

# --- 6. Train ---
print("Starting training...")
trainer_stats = trainer.train()
print("Training complete!")

# --- 7. Save Adapters ---
print(f"Saving LoRA adapters to '{new_model_name}'...")
model.save_pretrained(new_model_name)
tokenizer.save_pretrained(new_model_name)
print("Saved successfully. You can now run the Inference script.")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
Loading model: unsloth/gpt-oss-20b-unsloth-bnb-4bit...
==((====))==  Unsloth 2025.12.9: Fast Gpt_Oss patching. Transformers: 4.57.3.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.741 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.9.0+cu126. CUDA: 7.5. CUDA Toolkit: 12.6. Triton: 3.5.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.33.post2. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: Using float16 precision for gpt_oss won't work! Using float32.


model.safetensors.index.json: 0.00B [00:00, ?B/s]

model-00001-of-00004.safetensors:   0%|          | 0.00/4.00G [00:00<?, ?B/s]

model-00002-of-00004.safetensors:   0%|          | 0.00/4.00G [00:00<?, ?B/s]

model-00003-of-00004.safetensors:   0%|          | 0.00/3.37G [00:00<?, ?B/s]

...

Starting training...


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 15,530 | Num Epochs = 1 | Total steps = 300
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 4 x 1) = 4
 "-____-"     Trainable parameters = 7,962,624 of 20,922,719,808 (0.04% trained)


Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss
1,4.598500
2,3.752400
3,5.104500
4,4.117300
5,5.400600
6,4.073600
7,3.599900
8,4.232400
9,5.828600
10,4.351700


Training complete!
Saving LoRA adapters to 'my_finetuned_model_lora'...
Saved successfully. You can now run the Inference script.


In [4]:
from unsloth import FastLanguageModel, PatchDPOTrainer
from unsloth import is_bfloat16_supported
import torch
from datasets import load_dataset
from trl import DPOTrainer
from trl.trainer import DPOConfig

# --- 1. Configuration ---
# Point this to the folder where you saved your SFT adapters in Part 2
sft_adapter_path = "my_finetuned_model_lora"
new_dpo_model_name = "my_dpo_model_final"
max_seq_length = 512 # Reduced further to save VRAM
load_in_4bit = True

# --- 2. Patch DPO Trainer ---
# This fixes memory fragmentation issues automatically
PatchDPOTrainer()

# --- 3. Load the SFT Model ---
print(f"Loading SFT model from: {sft_adapter_path}")
# This automatically loads the Base Model + Your SFT Adapters
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = sft_adapter_path,
    max_seq_length = max_seq_length,
    load_in_4bit = load_in_4bit,
)

# --- 4. Setup DPO Format ---
# We use a simple chat template (Llama-3 style)
def formatting_func(example):
    # This function expects the dataset to have columns: 'prompt', 'chosen', 'rejected'
    return {
        "prompt": tokenizer.apply_chat_template([{"role": "user", "content": example["prompt"]}], tokenize=False, add_generation_prompt=True),
        "chosen": tokenizer.apply_chat_template([{"role": "assistant", "content": example["chosen"]}], tokenize=False, add_generation_prompt=False),
        "rejected": tokenizer.apply_chat_template([{"role": "assistant", "content": example["rejected"]}], tokenize=False, add_generation_prompt=False),
    }

# --- 5. Load Preference Dataset ---
# Since your medquad data lacks 'rejected' answers, we use a public mix for demonstration.
# To use your own, you must create a JSONL with keys: "prompt", "chosen", "rejected"
print("Loading DPO dataset...")
dataset = load_dataset("mlabonne/orpo-dpo-mix-40k", split="train").select(range(1000)) # Selecting 1k samples for speed
dataset = dataset.map(formatting_func, num_proc=2)

# --- 6. Configure DPO Trainer ---
dpo_trainer = DPOTrainer(
    model = model,
    ref_model = None, # Unsloth handles the reference model efficiently without loading it twice
    tokenizer = tokenizer,
    train_dataset = dataset,
    beta = 0.1, # The strength of the KL divergence penalty
    max_prompt_length = 256, # Adjust proportionally with max_seq_length
    max_length = max_seq_length,
    args = DPOConfig(
        per_device_train_batch_size = 1,
        gradient_accumulation_steps = 8, # Increased from 4 to further reduce memory
        warmup_steps = 10,
        max_steps = 100, # Short run for demo; increase for real training
        learning_rate = 5e-6, # DPO requires a very low LR
        fp16 = not is_bfloat16_supported(),
        bf16 = is_bfloat16_supported(),
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 42,
        output_dir = "dpo_output",
        report_to = "none",
    ),
)

# --- 7. Train ---
print("Starting DPO training...")
dpo_trainer.train()
print("DPO Training complete!")

# --- 8. Save Final Model ---
print(f"Saving DPO adapters to '{new_dpo_model_name}'...")
model.save_pretrained(new_dpo_model_name)
tokenizer.save_pretrained(new_dpo_model_name)
print("All done.")

Loading SFT model from: my_finetuned_model_lora
==((====))==  Unsloth 2025.12.9: Fast Gpt_Oss patching. Transformers: 4.57.3.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.741 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.9.0+cu126. CUDA: 7.5. CUDA Toolkit: 12.6. Triton: 3.5.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.33.post2. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: Using float16 precision for gpt_oss won't work! Using float32.


/usr/local/lib/python3.12/dist-packages/accelerate/utils/modeling.py:1566: UserWarning: Current model requires 16.0 bytes of buffer for offloaded layers, which seems does not fit any GPU's remaining memory. If you are experiencing a OOM later, please consider using offload_buffers=True.
  warnings.warn(


ValueError: Some modules are dispatched on the CPU or the disk. Make sure you have enough GPU RAM to fit the quantized model. If you want to dispatch the model on the CPU or the disk while keeping these modules in 32-bit, you need to set `llm_int8_enable_fp32_cpu_offload=True` and pass a custom `device_map` to `from_pretrained`. Check https://huggingface.co/docs/transformers/main/en/main_classes/quantization#offload-between-cpu-and-gpu for more details. 

In [1]:
import torch
from unsloth import FastLanguageModel
from unsloth.chat_templates import get_chat_template
import gradio as gr
from transformers import TextIteratorStreamer
from threading import Thread

# --- Configuration ---
adapter_path = "/content/model_output/checkpoint-300"
max_seq_length = 2048
load_in_4bit = True

# --- Load Model ---
print(f"Loading {adapter_path}...")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = adapter_path,
    max_seq_length = max_seq_length,
    load_in_4bit = load_in_4bit,
)
FastLanguageModel.for_inference(model)

tokenizer = get_chat_template(
    tokenizer,
    chat_template = "llama-3",
    mapping = {"role" : "role", "content" : "content", "user" : "user", "assistant" : "assistant"},
)

# --- Streaming Inference Function ---
def generate_response(message, history):
    messages = []
    for user_msg, bot_msg in history:
        messages.append({"role": "user", "content": user_msg})
        messages.append({"role": "assistant", "content": bot_msg})
    messages.append({"role": "user", "content": message})

    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize = True,
        add_generation_prompt = True,
        return_tensors = "pt",
    ).to("cuda")

    # Setup Streamer
    streamer = TextIteratorStreamer(tokenizer, skip_prompt=True, skip_special_tokens=True)

    # Generation settings
    generation_kwargs = dict(
        input_ids=inputs,
        streamer=streamer,
        max_new_tokens=256,
        use_cache=True,
        temperature=0.7,
        min_p=0.1,
    )

    # Run generation in a separate thread so the UI doesn't freeze
    thread = Thread(target=model.generate, kwargs=generation_kwargs)
    thread.start()

    # Yield output character by character
    partial_text = ""
    for new_text in streamer:
        partial_text += new_text
        yield partial_text

# --- Launch Gradio ---
demo = gr.ChatInterface(
    fn=generate_response,
    title="Medical Chatbot (Streaming)",
)
demo.launch(share=True)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
Loading /content/model_output/checkpoint-300...
==((====))==  Unsloth 2025.12.9: Fast Gpt_Oss patching. Transformers: 4.57.3.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.741 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.9.0+cu126. CUDA: 7.5. CUDA Toolkit: 12.6. Triton: 3.5.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.33.post2. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: Using float16 precision for gpt_oss won't work! Using float32.


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/gradio/chat_interface.py:347: UserWarning: The 'tuples' format for chatbot messages is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style 'role' and 'content' keys.
  self.chatbot = Chatbot(


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://813b970f568ee3547f.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [5]:
!zip -r model_output.zip /content/model_output

updating: content/model_output/ (stored 0%)
updating: content/model_output/checkpoint-300/ (stored 0%)
updating: content/model_output/checkpoint-300/adapter_model.safetensors (deflated 8%)
updating: content/model_output/checkpoint-300/special_tokens_map.json (deflated 71%)
updating: content/model_output/checkpoint-300/adapter_config.json (deflated 58%)
updating: content/model_output/checkpoint-300/tokenizer_config.json (deflated 89%)
updating: content/model_output/checkpoint-300/training_args.bin (deflated 53%)
updating: content/model_output/checkpoint-300/scheduler.pt (deflated 61%)
updating: content/model_output/checkpoint-300/optimizer.pt (deflated 100%)
updating: content/model_output/checkpoint-300/chat_template.jinja (deflated 67%)
updating: content/model_output/checkpoint-300/README.md (deflated 65%)
updating: content/model_output/checkpoint-300/trainer_state.json (deflated 86%)
updating: content/model_output/checkpoint-300/scaler.pt (deflated 64%)
updating: content/model_output/

In [ ]:
from google.colab import files
files.download('model_output.zip')

In [2]:
from unsloth import FastLanguageModel

# 1. Load your trained model (Step 2 or 3 output folder)
model_name = "/content/model_output/checkpoint-300" # Or "my_finetuned_model_lora"
print(f"Loading {model_name}...")

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = model_name,
    max_seq_length = 2048,
    load_in_4bit = True,
)

# 2. Save to GGUF (Quantized format)
# "q4_k_m" is the balanced standard (good speed/quality)
print("Saving to GGUF format... This might take a few minutes.")
model.save_pretrained_gguf("my_model_gguf", tokenizer, quantization_method = "q4_k_m")

print("Done! You now have a file like 'my_model_gguf/my_dpo_model_final-unsloth.Q4_K_M.gguf'")

Loading /content/model_output/checkpoint-300...
==((====))==  Unsloth 2025.12.9: Fast Gpt_Oss patching. Transformers: 4.57.3.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.741 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.9.0+cu126. CUDA: 7.5. CUDA Toolkit: 12.6. Triton: 3.5.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.33.post2. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: Using float16 precision for gpt_oss won't work! Using float32.


ValueError: Some modules are dispatched on the CPU or the disk. Make sure you have enough GPU RAM to fit the quantized model. If you want to dispatch the model on the CPU or the disk while keeping these modules in 32-bit, you need to set `llm_int8_enable_fp32_cpu_offload=True` and pass a custom `device_map` to `from_pretrained`. Check https://huggingface.co/docs/transformers/main/en/main_classes/quantization#offload-between-cpu-and-gpu for more details. 

In [ ]:
FROM ./my_dpo_model_final-unsloth.Q4_K_M.gguf
TEMPLATE """<|start_header_id|>user<|end_header_id|>

{{ .Prompt }}<|eot_id|><|start_header_id|>assistant<|end_header_id|>
"""

In [ ]:
#  Run command: ollama create mymedbot -f Modelfile

#  Run command: ollama run mymedbot (This will be much faster).